# Assignment 1 — 第 4 題：原始模型 SVD 分析

這份 Notebook 可獨立執行，分析 **原始 `meta-llama/Llama-3.2-1B`** 的 Layer **0、7、15** `q_proj` 權重。產出每層的完整奇異值與累積能量 CSV、至少保留 50% 能量的最小秩，以及作業要求的 **6 張圖**（各存 PNG 與 PDF）。

請在 Kaggle **Settings → Accelerator** 選 **GPU T4 x2**（單張 CUDA GPU 亦可），開啟 **Internet**，並在 **Add-ons → Secrets** 啟用標籤為 **`HF_TOKEN`** 的 Hugging Face Read 權杖。已有模型授權仍需把 Secret 授予這份 Notebook；Kaggle API token 與 `HF_TOKEN` 是兩種不同憑證。

選 **Run All** 開始。第一格先確認 GPU，之後才安裝套件及下載模型。輸出位置：`/kaggle/working/Assignment1/q4`。這份重構版本 **尚未重新執行 GPU 實驗**，所有 code cell output 都是空白，沒有預填或模擬結果。

來源：[作業官方程式](https://github.com/nycubigdata-2026fall/BigData2026-Assignment1)、本地繁體中文整理稿第 6.4 節。這題無須微調、資料集下載或準確率評估。


In [ ]:
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/Assignment1/q4')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STARTED = time.perf_counter()

def report_progress(stage, completed=0, total=None, status='running', **details):
    snapshot = {
        'run_stage': 'q4', 'stage': stage, 'status': status,
        'completed': completed, 'total': total,
        'phase_percent': round(100 * completed / total, 2) if total else None,
        'elapsed_seconds': round(time.perf_counter() - RUN_STARTED, 2),
        'updated_at_utc': datetime.now(timezone.utc).isoformat(), **details,
    }
    temporary = OUTPUT_DIR / 'progress.json.tmp'
    temporary.write_text(json.dumps(snapshot, indent=2), encoding='utf-8')
    temporary.replace(OUTPUT_DIR / 'progress.json')
    print('PROGRESS ' + json.dumps(snapshot), flush=True)

# 環境設定必須早於第一次 import torch / HF 套件。
os.environ.update(
    PYTORCH_ALLOC_CONF='expandable_segments:True', HF_HOME='/kaggle/temp/assignment1_hf',
    HF_HUB_DISABLE_TELEMETRY='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1',
    TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0',
    TRANSFORMERS_NO_ADVISORY_WARNINGS='1',
)
report_progress('q4/gpu_preflight', 0, 1)
import torch

TORCH_VERSION_BEFORE = torch.__version__
GPU_INFO = [
    {'index': index, 'name': torch.cuda.get_device_name(index),
     'total_memory_gib': round(torch.cuda.get_device_properties(index).total_memory / 1024**3, 2)}
    for index in range(torch.cuda.device_count())
]
BOOTSTRAP_REPORT = {
    'cuda_allocator_config': os.environ['PYTORCH_ALLOC_CONF'],
    'torch_version': TORCH_VERSION_BEFORE, 'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(), 'gpu_count': len(GPU_INFO),
    'gpu_names': [gpu['name'] for gpu in GPU_INFO], 'gpus': GPU_INFO,
}
(OUTPUT_DIR / 'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP_REPORT, indent=2), encoding='utf-8')
if not BOOTSTRAP_REPORT['cuda_available']:
    report_progress('q4/gpu_preflight', 0, 1, status='error', **BOOTSTRAP_REPORT)
    raise RuntimeError('CUDA GPU 尚未啟用：在 Kaggle Settings 選 GPU T4 x2，再 Restart Session / Run All。')
if tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2]) < (2, 1):
    report_progress('q4/gpu_preflight', 0, 1, status='error', reason='torch_version_too_old')
    raise RuntimeError('需要 Kaggle 已提供 PyTorch >= 2.1；請切換目前 GPU 映像。')
report_progress('q4/gpu_preflight', 1, 1, **BOOTSTRAP_REPORT)


## 固定套件版本

使用官方 PyPI 套件：`transformers==4.56.2`、`datasets==4.1.1`、`accelerate==1.10.1`、`huggingface_hub==0.34.4`。`datasets` 與第 1–3 題版本一致，本題本身不載入資料集。官方依賴中 `transformers` 要求 Hub `>=0.34.0,<1.0`；上述版本符合。

沿用 Kaggle 已提供的 GPU PyTorch，將其現有版本寫入 pip constraint，避免安裝其他套件時替換 PyTorch；這與成功的 Version 13 GPU bootstrap 相同。若目前 Session 已載入其他版本的 Hugging Face 套件，安裝後程式會要求 **Restart Session**；重新開啟 Session 再 Run All 即可。


In [ ]:
import importlib.metadata as package_metadata
import subprocess

PINNED_PACKAGES = {
    'transformers': '4.56.2', 'datasets': '4.1.1',
    'accelerate': '1.10.1', 'huggingface_hub': '0.34.4',
}
old_loaded_versions = {
    name: getattr(sys.modules[name], '__version__', None)
    for name in PINNED_PACKAGES if name in sys.modules
}
constraint_path = Path('/kaggle/temp/assignment1_q4_torch_constraint.txt')
constraint_path.parent.mkdir(parents=True, exist_ok=True)
constraint_path.write_text('torch==' + TORCH_VERSION_BEFORE + '\n', encoding='utf-8')
requirements = [f'{name}=={version}' for name, version in PINNED_PACKAGES.items()]
requirements += ['safetensors', 'numpy<3', 'pandas<3', 'matplotlib<4']
report_progress('q4/package_install', 0, 1)
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
     '--timeout', '20', '--retries', '1', '--constraint', str(constraint_path),
     '--index-url', 'https://pypi.org/simple', *requirements],
    text=True, encoding='utf-8', errors='replace', capture_output=True,
)
if install_result.returncode != 0:
    # 遮罩原始 pip 輸出，避免額外設定的鏡像/連線憑證出現在 output。
    report_progress('q4/package_install', 0, 1, status='error', pip_return_code=install_result.returncode)
    raise RuntimeError('官方 PyPI 套件安裝未成功。請確認 Kaggle Internet 已開啟。')
versions_after_install = {name: package_metadata.version(name) for name in PINNED_PACKAGES}
if (versions_after_install != PINNED_PACKAGES
        or package_metadata.version('torch') != TORCH_VERSION_BEFORE
        or any(version != PINNED_PACKAGES[name] for name, version in old_loaded_versions.items())):
    report_progress('q4/package_install', 0, 1, status='error', reason='restart_required')
    raise RuntimeError('套件版本或 Session 中已載入的版本不一致，請 Restart Session 再 Run All。')
report_progress('q4/package_install', 1, 1, packages=versions_after_install, torch_version=TORCH_VERSION_BEFORE)


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from huggingface_hub import HfApi
from huggingface_hub.utils import logging as hub_logging
from transformers import AutoModelForCausalLM
from transformers.utils import logging as transformers_logging

hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
MODEL_ID = 'meta-llama/Llama-3.2-1B'
MODEL_REVISION = '4e20de362430cd3b72f300e6b0f18e50e7166e08'
MODEL_DTYPE = torch.float16
LAYER_INDICES = (0, 7, 15)
PLOT_LIMIT = 200
ENERGY_THRESHOLD_PERCENT = 50.0
ENVIRONMENT = {
    'python': sys.version.split()[0], 'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda, 'cuda_allocator_config': os.environ['PYTORCH_ALLOC_CONF'],
    'packages': {name: package_metadata.version(name) for name in
                 (*PINNED_PACKAGES, 'numpy', 'pandas', 'matplotlib')},
    'gpus': GPU_INFO, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'model_weight_dtype': 'float16', 'svd_input_dtype': 'float32',
}
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')


## 讀取 Kaggle Secret 並載入原始模型

程式只在函式內取得 `HF_TOKEN`，明確傳入模型下载呼叫，不呼叫 `login()`、不把權杖寫入檔案或顯示在 output。Secret、模型授權或網路錯誤會轉成不含原始例外內容的訊息。進度僅記錄 Secret 服務是否成功與固定模型版本，不記錄權杖。

模型使用與第 1–3 題已完成實驗相同的 commit **`4e20de362430cd3b72f300e6b0f18e50e7166e08`**，以 **FP16** 載入 `cuda:0`。本題只讀原始模型，沒有微調、adapter 或量化；SVD 前將各層 `q_proj` 權重轉為 **float32**，與官方範例的 `q_weight.float()` 相同。


In [ ]:
def load_original_model_securely():
    hf_token = None
    report_progress('q4/hf_secret', 0, 1)
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError('EmptySecret')
    except Exception:
        report_progress('q4/hf_secret', 0, 1, status='error', reason='secret_unavailable')
        raise RuntimeError('無法讀取 HF_TOKEN。請在 Kaggle Add-ons → Secrets 啟用此 Secret。') from None
    try:
        report_progress('q4/hf_secret', 1, 1, secret_service='kaggle_secrets.UserSecretsClient',
                        secret_access_verified=True)
        report_progress('q4/hf_access', 0, 1)
        revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, token=hf_token).sha
        if revision != MODEL_REVISION:
            raise RuntimeError('RevisionMismatch')
        report_progress('q4/hf_access', 1, 1, model_id=MODEL_ID, model_revision=revision,
                        fixed_revision_access_verified=True)
        report_progress('q4/model_load', 0, 1, model_revision=revision)
        loaded_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, token=hf_token, dtype=MODEL_DTYPE,
            device_map={'': 'cuda:0'}, low_cpu_mem_usage=True,
            attn_implementation='eager', trust_remote_code=False,
        )
        loaded_model.eval()
        loaded_model.requires_grad_(False)
        return loaded_model
    except Exception:
        report_progress('q4/model_access_or_load', 0, 1, status='error', reason='masked_hf_or_model_error')
        raise RuntimeError('原始 Llama-3.2-1B 載入失敗。請確認 Internet、HF_TOKEN、模型授權及 GPU 記憶體。原始錯誤已遮罩。') from None
    finally:
        hf_token = None

model = load_original_model_securely()
if (model.config.model_type != 'llama' or len(model.model.layers) <= max(LAYER_INDICES)
        or any(model.model.layers[index].self_attn.q_proj.weight.dtype != MODEL_DTYPE for index in LAYER_INDICES)):
    report_progress('q4/model_load', 0, 1, status='error', reason='model_architecture_or_dtype_mismatch')
    raise RuntimeError('模型架構或 q_proj FP16 型別不符，停止分析。')
report_progress('q4/model_load', 1, 1, model_id=MODEL_ID, model_revision=MODEL_REVISION,
                number_of_layers=len(model.model.layers), model_weight_dtype='float16')


## 對完整權重矩陣計算 SVD 與累積能量

每層均讀取 `model.model.layers[k].self_attn.q_proj.weight`，在 **float32** 使用 `torch.linalg.svdvals` 取得**全部**奇異值。它與 SVD 中的 `S` 相同，本題不需儲存 `U` 或 `Vh`。`driver='gesvd'` 使用 CUDA 的完整 SVD；不使用截斷、隨機近似或預先設定 target rank。

對由大到小排列的奇異值 $\sigma_i$，累積能量為

$$E(r)=100\,\frac{\sum_{i=1}^{r}\sigma_i^2}{\sum_{i=1}^{n}\sigma_i^2}.$$

SVD 使用 float32，平方與累加改用 float64 降低數值累加誤差。**分母始終使用全部奇異值**；前 200 個只限制畫圖範圍。最小秩為 $\min\{r:E(r)\ge 50\%\}$，rank 採 1 開始計數。程式會檢查前一個 rank 小於 50%，確保結果是最小值。


In [ ]:
def spectrum_statistics(singular_values):
    if (singular_values.ndim != 1 or not len(singular_values)
            or not np.isfinite(singular_values).all() or np.any(singular_values < 0)
            or np.any(np.diff(singular_values) > 0)):
        raise RuntimeError('奇異值譜必須有限、非負且由大到小排列。')
    squared_values = singular_values ** 2
    total_energy = float(squared_values.sum(dtype=np.float64))
    if not np.isfinite(total_energy) or total_energy <= 0:
        raise RuntimeError('完整奇異值譜總能量無效。')
    energy_percent = np.cumsum(squared_values, dtype=np.float64) / total_energy * 100.0
    rank_50 = int(np.searchsorted(energy_percent, ENERGY_THRESHOLD_PERCENT, side='left')) + 1
    if not (rank_50 <= len(energy_percent) and energy_percent[rank_50 - 1] >= ENERGY_THRESHOLD_PERCENT
            and (rank_50 == 1 or energy_percent[rank_50 - 2] < ENERGY_THRESHOLD_PERCENT)
            and np.isclose(energy_percent[-1], 100.0, rtol=0, atol=1e-8)):
        raise RuntimeError('完整累積能量或 50% 最小秩驗證失敗。')
    return squared_values, total_energy, energy_percent, rank_50

# CPU self-check only; synthetic values are never saved as model results.
_, test_total, test_energy, test_rank = spectrum_statistics(np.ones(2048))
assert test_total == 2048 and test_rank == 1024 and test_energy[199] == 100 * 200 / 2048
assert spectrum_statistics(np.ones(2))[-1] == 1  # Exact 50% counts as retained.
spectra = {}
summary_rows = []
started_at_utc = datetime.now(timezone.utc).isoformat()
with torch.inference_mode():
    for layer_position, layer_index in enumerate(LAYER_INDICES, start=1):
        report_progress('q4/svd', layer_position - 1, len(LAYER_INDICES), active_layer=layer_index)
        q_weight = model.model.layers[layer_index].self_attn.q_proj.weight.detach()
        weight = q_weight.float().contiguous()
        weight_shape = tuple(weight.shape)
        if weight.ndim != 2 or weight.device.type != 'cuda' or not bool(torch.isfinite(weight).all().item()):
            raise RuntimeError(f'Layer {layer_index} 的 q_proj 必須為有限的 CUDA 二維矩陣。')
        torch.cuda.synchronize()
        svd_started = time.perf_counter()
        try:
            singular_tensor = torch.linalg.svdvals(weight, driver='gesvd')
        except Exception as exc:
            report_progress('q4/svd', layer_position - 1, len(LAYER_INDICES), status='error',
                            active_layer=layer_index, error_type=type(exc).__name__)
            raise
        torch.cuda.synchronize()
        svd_seconds = time.perf_counter() - svd_started
        singular_values = singular_tensor.cpu().double().numpy()
        if len(singular_values) != min(weight_shape):
            raise RuntimeError(f'Layer {layer_index} 未取得完整奇異值譜。')
        squared_values, total_energy, energy_percent, rank_50 = spectrum_statistics(singular_values)
        spectra[layer_index] = {'singular_values': singular_values, 'energy_percent': energy_percent}
        pd.DataFrame({
            'rank': np.arange(1, len(singular_values) + 1), 'singular_value': singular_values,
            'squared_singular_value': squared_values, 'cumulative_energy_percent': energy_percent,
        }).to_csv(OUTPUT_DIR / f'layer_{layer_index}_q_proj_full_spectrum.csv', index=False)
        summary_rows.append({
            'layer': layer_index, 'original_weight_dtype': 'float16', 'svd_input_dtype': 'float32',
            'weight_rows': weight_shape[0], 'weight_columns': weight_shape[1],
            'full_singular_value_count': len(singular_values), 'minimum_rank_for_50_percent': rank_50,
            'energy_at_rank_50_percent': float(energy_percent[rank_50 - 1]),
            'energy_before_rank_50_percent': float(energy_percent[rank_50 - 2]) if rank_50 > 1 else 0.0,
            'energy_at_rank_200_percent': float(energy_percent[min(PLOT_LIMIT, len(singular_values)) - 1]),
            'total_squared_singular_value_energy': total_energy, 'svd_seconds': svd_seconds,
        })
        report_progress('q4/svd', layer_position, len(LAYER_INDICES), active_layer=layer_index,
                        full_singular_value_count=len(singular_values), minimum_rank_for_50_percent=rank_50,
                        svd_seconds=round(svd_seconds, 3))
        del q_weight, weight, singular_tensor
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'q4_rank50_summary.csv', index=False)
display(summary)
del model
gc.collect()
torch.cuda.empty_cache()


## 產出作業要求的 6 張圖

每層各產出一張**奇異值大小長條圖**與一張**累積能量折線圖**；橫軸僅涵蓋前 200 個奇異值 / rank。圖表文字採英文。折線圖的分母仍是全部奇異值的平方和，50% 虛線與最小 rank 註記以完整譜計算；若最小 rank 超過 200，只在圖內文字說明其值，不延伸圖表範圍。


In [ ]:
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'savefig.dpi': 220})
figure_files = []
report_progress('q4/plots', 0, 6)

def save_figure(fig, layer_index, kind):
    fig.tight_layout()
    for extension in ('png', 'pdf'):
        name = f'layer_{layer_index}_{kind}_first200.{extension}'
        fig.savefig(OUTPUT_DIR / name, bbox_inches='tight')
        figure_files.append(name)
    plt.show()
    plt.close(fig)
    report_progress('q4/plots', len(figure_files) // 2, 6, active_layer=layer_index)

for layer_index, row in zip(LAYER_INDICES, summary_rows):
    spectrum = spectra[layer_index]
    n_plot = min(PLOT_LIMIT, len(spectrum['singular_values']))
    ranks = np.arange(1, n_plot + 1)
    rank_50 = row['minimum_rank_for_50_percent']
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.bar(ranks, spectrum['singular_values'][:n_plot], width=0.85, color='#2667A1')
    ax.set(title=f'Layer {layer_index} q_proj: First {n_plot} Singular Values',
           xlabel='Singular Value Index (1-based)', ylabel='Singular Value Magnitude', xlim=(0.5, n_plot + 0.5))
    ax.grid(axis='y', alpha=0.25)
    save_figure(fig, layer_index, 'singular_values')
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.plot(ranks, spectrum['energy_percent'][:n_plot], color='#237C50', linewidth=2)
    ax.axhline(ENERGY_THRESHOLD_PERCENT, linestyle='--', color='#A84735', label='50% energy threshold')
    if rank_50 <= n_plot:
        ax.axvline(rank_50, linestyle=':', color='#A84735', label=f'Minimum rank = {rank_50}')
        ax.scatter([rank_50], [spectrum['energy_percent'][rank_50 - 1]], color='#A84735', zorder=3)
    else:
        ax.text(0.03, 0.94, f'Minimum rank at 50%: {rank_50} (outside shown range)',
                transform=ax.transAxes, va='top')
    ax.set(title=f'Layer {layer_index} q_proj: Cumulative Energy (First {n_plot} Ranks)',
           xlabel='Target Rank (1-based)', ylabel='Cumulative Energy (%)', xlim=(1, n_plot), ylim=(0, 100))
    ax.grid(alpha=0.25)
    ax.legend(loc='lower right')
    save_figure(fig, layer_index, 'cumulative_energy')


In [ ]:
required_csv_files = [f'layer_{index}_q_proj_full_spectrum.csv' for index in LAYER_INDICES]
required_csv_files += ['q4_rank50_summary.csv']
expected_files = required_csv_files + figure_files + ['bootstrap_report.json', 'environment.json', 'progress.json']
if len(figure_files) != 12 or not all((OUTPUT_DIR / name).is_file() and (OUTPUT_DIR / name).stat().st_size > 0 for name in expected_files):
    report_progress('q4/output_validation', 0, 1, status='error', reason='missing_or_empty_output')
    raise RuntimeError('Q4 輸出檔案不完整；需要完整 CSV、6 張 PNG 與相同 6 張 PDF。')
results = {
    'status': 'completed_from_original_model_weights', 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'started_at_utc': started_at_utc, 'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_weight_dtype': 'float16', 'svd_input_dtype': 'float32',
    'svd_method': 'torch.linalg.svdvals on each full q_proj matrix, CUDA gesvd, float32',
    'energy_accumulation_dtype': 'float64',
    'energy_formula': '100 * cumsum(all_singular_values ** 2) / sum(all_singular_values ** 2)',
    'energy_threshold_percent': ENERGY_THRESHOLD_PERCENT, 'plotted_singular_values_per_layer': PLOT_LIMIT,
    'plot_energy_denominator': 'all_squared_singular_values', 'layers': summary.to_dict(orient='records'),
    'figure_files': figure_files, 'environment': ENVIRONMENT, 'bootstrap_report': BOOTSTRAP_REPORT,
}
(OUTPUT_DIR / 'q4_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
expected_files += ['q4_results.json']
report_progress('q4/completed', len(LAYER_INDICES), len(LAYER_INDICES), status='completed',
                completed_figures=len(figure_files) // 2, output_file_count=len(expected_files))
print('Q4 實際分析完成。輸出檔案：', *expected_files, 'Output directory:', str(OUTPUT_DIR), sep='\n', flush=True)


## 保存與下載結果

執行完成後使用 Kaggle **Save Version / Save & Run All** 保存 Notebook 與 `/kaggle/working/Assignment1/q4` 內容；可從該版本 **Output** 下載結果。`q4_rank50_summary.csv` 的 `minimum_rank_for_50_percent` 是第 4(b) 題的數值，`q4_results.json` 保存固定模型 commit、環境及真實執行摘要；每層 `full_spectrum.csv` 保留全部奇異值，而非僅前 200 個。

執行中 `PROGRESS` 會即時顯示階段與已完成數量，`progress.json` 保存最新狀態，包含 UTC 更新時間。`bootstrap_report.json` 保存實際 CUDA/GPU 證據；`q4_results.json` 只在所有 CSV 與 6 張 PNG、6 張 PDF 產出後寫入完成狀態。模型下載快取保存在 `/kaggle/temp`，不混入本題輸出。
